In [ ]:
# CELL 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DFU_PATH = '/content/drive/MyDrive/DFU'
print('Drive mounted and path changed')

In [ ]:
# CELL 2 — Copy dataset to local disk (faster I/O than Drive)
import shutil,os
from pathlib import Path

LOCAL_DATA = '/content/dfu_data'
if os.path.exists(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)

shutil.copytree(DRIVE_DFU_PATH, LOCAL_DATA)

In [ ]:
# CELL 3 — Auto-detect class folders
ULCER_ALIASES  = {'abnormal','abnormal(ulcer)','ulcer','infected'}
NORMAL_ALIASES = {'normal','normal(healthyskin)','normal(healthy_skin)','healthy'}
VALID_EXTS     = {'.jpg','.jpeg','.png','.bmp'}

def find_class_dirs(root):
    result = {}
    for d in Path(root).rglob('*'):
        if not d.is_dir(): continue
        key = d.name.lower().replace(' ','').replace('-','_')
        imgs = [f for f in d.iterdir() if f.suffix.lower() in VALID_EXTS]
        if not imgs: continue
        if key in {a.replace(' ','').replace('-','_') for a in ULCER_ALIASES}:
            result['Abnormal'] = d
        elif key in {a.replace(' ','').replace('-','_') for a in NORMAL_ALIASES}:
            result['Normal'] = d
    return result

class_dirs = find_class_dirs(LOCAL_DATA)
assert len(class_dirs) == 2, f'Expected 2 classes, found: {list(class_dirs.keys())}'

for label, path in class_dirs.items():
    n = len([f for f in path.rglob('*') if f.suffix.lower() in VALID_EXTS])
    print(f'  {label:12s}: {n} images  ({path.name})')

In [ ]:
# CELL 4 — Train/val split (85/15)
import numpy as np
SPLIT_DIR = '/content/split'
rng = np.random.default_rng(42)
if os.path.exists(SPLIT_DIR): shutil.rmtree(SPLIT_DIR)

for label, src in class_dirs.items():
    files = sorted(f for f in src.rglob('*') if f.suffix.lower() in VALID_EXTS)
    rng.shuffle(files := np.array(files))
    n_val = max(1, int(len(files) * 0.15))
    for split, subset in [('val', files[:n_val]), ('train', files[n_val:])]:
        dst = Path(SPLIT_DIR) / split / label
        dst.mkdir(parents=True, exist_ok=True)
        for f in subset: shutil.copy2(f, dst / f.name)
    print(f'{label}: {len(files)-n_val} train | {n_val} val')

In [ ]:
# CELL 5 — Data generators with EfficientNet's preprocess_input (scales to [-1,1])
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE, BATCH_SIZE = (224, 224), 16

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20, width_shift_range=0.15, height_shift_range=0.15,
    zoom_range=0.15, brightness_range=[0.85, 1.15],
    horizontal_flip=True, fill_mode='nearest'
).flow_from_directory(f'{SPLIT_DIR}/train', target_size=IMG_SIZE,
                      batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)

val_gen = ImageDataGenerator(preprocessing_function=preprocess_input
).flow_from_directory(f'{SPLIT_DIR}/val', target_size=IMG_SIZE,
                      batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

CLASS_NAMES  = list(train_gen.class_indices.keys())
NUM_CLASSES  = len(CLASS_NAMES)
cw_arr       = compute_class_weight('balanced', classes=np.unique(train_gen.classes), y=train_gen.classes)
class_weights = dict(enumerate(cw_arr))
print(f'Classes: {train_gen.class_indices} | Weights: {class_weights}')



In [ ]:
# CELL 6 — Build EfficientNetB3 with frozen backbone (Stage 1: train head only)
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras import layers, Model
import tensorflow as tf

base = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
base.trainable = False

x   = layers.GlobalAveragePooling2D()(base.output)
x   = layers.BatchNormalization()(x)
x   = layers.Dense(128, activation='relu')(x)
x   = layers.Dropout(0.4)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(base.input, out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# CELL 7 — Stage 1: Train head with backbone frozen
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

SAVE_PATH = '/content/drive/MyDrive/DFU/model.keras'

h1 = model.fit(
    train_gen, epochs=15, validation_data=val_gen, class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
        ModelCheckpoint(SAVE_PATH, monitor='val_accuracy', save_best_only=True),
    ]
)
print(f'Stage 1 best val_accuracy: {max(h1.history["val_accuracy"]):.4f}')

In [ ]:
# CELL 8 — Stage 2: Fine-tune top 20 backbone layers at lower LR
from tensorflow.keras.models import load_model

model = load_model(SAVE_PATH)

for layer in model.layers[:-24]:
    layer.trainable = False
for layer in model.layers[-24:]:
    layer.trainable = True
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(5e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])

h2 = model.fit(
    train_gen, epochs=30, validation_data=val_gen, class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7),
        ModelCheckpoint(SAVE_PATH, monitor='val_accuracy', save_best_only=True),
    ]
)
print(f'Stage 2 best val_accuracy: {max(h2.history["val_accuracy"]):.4f}')

In [ ]:
# CELL 9 — Evaluation
from sklearn.metrics import classification_report
from tensorflow.keras.models import load_model

best = load_model('/content/drive/MyDrive/DFU/model.keras')
val_gen.reset()
preds  = best.predict(val_gen, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = val_gen.classes

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:

# CELL 10 — Save weights + download
from google.colab import files

model.save_weights('/content/drive/MyDrive/DFU/model_weights.weights.h5')
files.download('/content/drive/MyDrive/DFU/model_weights.weights.h5')
files.download('/content/drive/MyDrive/DFU/class_info.json')